# Aligning two Visium sections to each other

Every other notebook in `examples/` aligns MSI onto Visium -- two different technologies. This
one aligns **two Visium sections to each other** (e.g. two consecutive mouse brain sections,
or a treated/control pair), using exactly the same `sw.align()` call. `spatialwarp` doesn't know
or care that both sides happen to be Visium this time: it just needs two
`spatialdata.SpatialData` objects, each with one image and one table whose `obsm['spatial']`
matches that image's pixel space.

Once registered, every spot in the `fixed` section gets the gene expression of its nearest
matched spot in the `moving` section merged in, in `obsm['ST48_expression']` alongside its own
expression -- handy for directly comparing the same tissue location across two sections
(consecutive sections, replicates, conditions, ...) without having to re-cluster/re-annotate.

In [1]:
from visium_utils import load_classic_visium_hires  # examples/visium_utils.py

import spatialwarp as sw

/home/croizer/.local/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/croizer/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
/home/croizer/.local/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


A couple of steps below open a window where you click points with your mouse. For that to
work in Jupyter, run this cell first to switch to a real window backend instead of the default
static-image mode (`tk` works out of the box with no extra install; use `qt` instead if you
have PyQt5/PySide installed and prefer it).

In [2]:
%matplotlib tk

## Step 1 -- Load the Visium sections

Neither of these Space Ranger runs saved a full-resolution H&E image, only the downscaled hires
preview -- `load_classic_visium_hires` (see `examples/visium_utils.py`) rescales each section's
spot coordinates into its own hires image's pixel space using Space Ranger's own scale factor,
then packages image + points as a plain SpatialData object (same helper used in
`examples/msi_to_classic_visium.ipynb`). The counts files here are named with a dataset-id
prefix rather than Space Ranger's default `filtered_feature_bc_matrix.h5`, hence `counts_file=`.

`ST50` is loaded here too but only used starting at Step 4 below -- Steps 1-3 first walk through
the ST48-vs-ST49 pair on its own.

In [ ]:
st48_sdata, st48_he = load_classic_visium_hires(
    "/home/croizer/Documents/01_Ressources/02_Public_Visium/brain/mice/cell2loc/ST48",
    dataset_id="ST48",
    counts_file="ST8059048_filtered_feature_bc_matrix.h5",
)
st49_sdata, st49_he = load_classic_visium_hires(
    "/home/croizer/Documents/01_Ressources/02_Public_Visium/brain/mice/cell2loc/ST49",
    dataset_id="ST49",
    counts_file="ST8059049_filtered_feature_bc_matrix.h5",
)
st50_sdata, st50_he = load_classic_visium_hires(
    "/home/croizer/Documents/01_Ressources/02_Public_Visium/brain/mice/cell2loc/ST50",
    dataset_id="ST50",
    counts_file="ST8059050_filtered_feature_bc_matrix.h5",
)
st48_sdata, st49_sdata, st50_sdata

## Step 2 -- Register the two H&E images

Click a point on the left (ST48), then its match on the right (ST49); repeat for ~5-10
well-spread landmarks (tissue corners/folds/vessels are good choices), then close the window.

In [4]:
moving_landmarks, fixed_landmarks, _, _ = sw.pick_landmarks(
    st48_he, st49_he, output_csv="landmarks_ST48_ST49.csv"
)

can't invoke "event" command: application has been destroyed
    while executing
"event generate $w <<ThemeChanged>>"
    (procedure "ttk::ThemeChanged" line 6)
    invoked from within
"ttk::ThemeChanged"


In [5]:
registration_result = sw.register_elastic(
    moving_image=st48_he,
    fixed_image=st49_he,
    moving_landmarks=moving_landmarks,
    fixed_landmarks=fixed_landmarks,
    mesh_size=(8, 8),
    number_of_iterations=100,
)
registration_result.save("registration_ST48_ST49")  # sw.RegistrationResult.load(...) to reuse later

QC: warp ST48's image onto ST49's grid and compare -- tissue outlines/folds should line up.

In [6]:
import matplotlib.pyplot as plt

warped_st48 = registration_result.warp_image(st48_he)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(st49_he)
axes[0].set_title("ST49 (fixed)")
axes[1].imshow(warped_st48)
axes[1].set_title("ST48 warped onto ST49's grid")
plt.show()

## Step 3 -- Match spots and merge expression

For every ST49 spot, find its nearest matched ST48 spot and merge that spot's full gene
expression into `obsm["ST48_expression"]`. Both sections are whole-transcriptome here (~31,000
genes), so this densifies a `(n_matched, n_genes)` matrix -- fine at this size, but for much
larger gene panels you may want to subset `moving`'s table to a smaller gene set (e.g. highly
variable genes) before calling `sw.align()` if memory becomes a concern.

`distance_threshold` is in ST49's hires-image pixels; loosen it if too few spots match, tighten
it if matches look too far apart on the QC plot below.

In [7]:
merged = sw.align(
    moving=st48_sdata,
    fixed=st49_sdata,
    registration_result=registration_result,
    distance_threshold=15.0,
    moving_table_key="visium",
    fixed_table_key="visium",
    moving_obsm_key="ST48_expression",
)
print(f"kept {merged.n_obs} of {st49_sdata.tables['visium'].n_obs} ST49 spots")
merged

kept 3373 of 3499 ST49 spots


AnnData object with n_obs × n_vars = 3373 × 31053
    obs: 'in_tissue', 'array_row', 'array_col', 'spot_id', 'region', 'nearest_index', 'nearest_distance'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial', 'spatialdata_attrs', 'spatialwarp'
    obsm: 'spatial', 'spatial_warped', 'ST48_expression'

**Check it worked:** two sanity checks that don't require picking a specific marker gene ahead
of time.

1. Spatial: the warped ST48 points (`obsm['spatial_warped']`) should trace out ST49's own tissue
   shape, not a scattered cloud.
2. Signal: each ST49 spot's own total transcript count should correlate with its matched ST48
   spot's total count -- two sections of the same tissue should broadly agree on where RNA
   density is high/low (e.g. white vs. grey matter), even though the two spot grids don't align
   spot-for-spot.

In [8]:
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].scatter(*merged.obsm["spatial_warped"].T, s=3, alpha=0.4)
axes[0].invert_yaxis()
axes[0].set_aspect("equal")
axes[0].set_title("ST48 spots warped into ST49 space")

st49_total = np.asarray(merged.X.sum(axis=1)).ravel()
st48_total = merged.obsm["ST48_expression"].sum(axis=1).values
axes[1].scatter(st48_total, st49_total, s=4, alpha=0.4)
axes[1].set_xlabel("ST48 (matched) total counts")
axes[1].set_ylabel("ST49 total counts")
axes[1].set_title("Per-spot total-count agreement")
plt.tight_layout()
plt.show()

import scipy.stats
r, p = scipy.stats.pearsonr(st48_total, st49_total)
print(f"ST48 vs ST49 total counts: r={r:.2f}, p={p:.2e}")

ST48 vs ST49 total counts: r=0.42, p=2.78e-146


## Step 4 -- Add a third section (ST50)

Same recipe as Steps 2-3, repeated for ST50 against ST49: pick landmarks, register, match and
merge. ST49 stays the common reference both times, but ST50's own H&E is a different image from
ST48's, so it needs its own registration -- there's no shared transform to reuse here (unlike
`examples/l12_walkthrough.ipynb`'s Step 4, where two MSI modalities share one H&E and so share
one registration against Visium).

`merged50`'s matched set of ST49 spots won't necessarily be identical to `merged`'s (each
registration/match can keep slightly different spots), so `ST50_expression` is reindexed onto
`merged`'s existing spots, leaving `NaN` wherever ST50 didn't have a close-enough match -- same
pattern as the lipidomics step in `examples/l12_walkthrough.ipynb`.

In [ ]:
moving_landmarks_50, fixed_landmarks_50, _, _ = sw.pick_landmarks(
    st50_he, st49_he, output_csv="landmarks_ST50_ST49.csv"
)

In [ ]:
registration_result_50 = sw.register_elastic(
    moving_image=st50_he,
    fixed_image=st49_he,
    moving_landmarks=moving_landmarks_50,
    fixed_landmarks=fixed_landmarks_50,
    mesh_size=(8, 8),
    number_of_iterations=100,
)
registration_result_50.save("registration_ST50_ST49")

merged50 = sw.align(
    moving=st50_sdata,
    fixed=st49_sdata,
    registration_result=registration_result_50,
    distance_threshold=15.0,
    moving_table_key="visium",
    fixed_table_key="visium",
    moving_obsm_key="ST50_expression",
)
print(f"kept {merged50.n_obs} of {st49_sdata.tables['visium'].n_obs} ST49 spots (ST50 match)")

merged.obsm["ST50_expression"] = merged50.obsm["ST50_expression"].reindex(merged.obs_names)
merged.obs["nearest_distance_ST50"] = merged50.obs["nearest_distance"].reindex(merged.obs_names)
print(f"ST50_expression missing for {merged.obsm['ST50_expression'].isna().any(axis=1).sum()} of {merged.n_obs} spots")
merged

## Bonus: same gene, all three sections, RGB additive overlay

Since all three sections are the same technology (whole mouse transcriptome), `merged.X` (ST49's
own expression), `merged.obsm["ST48_expression"]`, and `merged.obsm["ST50_expression"]` all share
the same gene set -- so a single gene can be compared directly, spot-for-spot, at each matched
location. `gene` defaults to `"Mbp"` (myelin basic protein, a strong white-matter marker with an
easily recognizable spatial pattern in a brain section) -- swap in any gene present in all three.

Plotted like a three-channel immunofluorescence overlay: ST48 in red, ST49 in green, ST50 in
blue, at ST49's own spot positions. White/pale regions = all three sections agree the gene is
high there; a saturated single color = that section disagrees with the other two (could be
genuine biological variation, a registration/matching error, or just per-spot noise). Spots ST50
had no close match for (`NaN` in `ST50_expression`) are treated as 0 in the blue channel, i.e.
they show as whatever red/green alone would give.

In [ ]:
from matplotlib.patches import Patch
import scipy.sparse as sp


def _dense_1d(x):
    return x.toarray().ravel() if sp.issparse(x) else np.asarray(x).ravel()


gene = "Lamp5"

channels = {
    "ST48": ("red", merged.obsm["ST48_expression"][gene].values),
    "ST49": ("green", _dense_1d(merged[:, gene].X)),
    "ST50": ("blue", merged.obsm["ST50_expression"][gene].fillna(0).values),
}
color_idx = {"red": 0, "green": 1, "blue": 2}

spatial_coords = merged.obsm["spatial"]
rgb_image = np.zeros((merged.n_obs, 3))

for label, (color, values) in channels.items():
    values = values.astype(float)
    vmax = np.nanpercentile(values, 99)
    rgb_image[:, color_idx[color]] = np.clip(values / vmax, 0, 1) if vmax > 0 else values

fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(spatial_coords[:, 0], -spatial_coords[:, 1], c=rgb_image, s=60, alpha=0.9)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(f"{gene}: ST48 vs ST49 vs ST50 (RGB additive overlay)")

legend_elements = [Patch(facecolor=color, label=label) for label, (color, _) in channels.items()]
ax.legend(handles=legend_elements, loc="upper right", frameon=True)
plt.tight_layout()
plt.show()

For contrast, here's the same additive overlay using each section's `array_row`/`array_col` --
the fixed hardware spot-grid indices Space Ranger assigns, identical in layout across every
capture area of the same slide format. Unlike raw pixel coordinates (arbitrary per capture area,
and not shown here for that reason), these genuinely are "the same coordinates" already, with no
registration involved at all.

Careful what this does and doesn't show: matching up here means two sections' spots landed in
the **same physical capture-area slot**, not the same **tissue location** -- each section's
tissue was placed independently on its own capture area, so overlap by array position is
essentially coincidental with respect to anatomy. This is the honest "no alignment, and not even
trying to align" baseline that Steps 2-4's registration is solving.

In [ ]:
st48_obs = st48_sdata.tables["visium"].obs
st49_obs = st49_sdata.tables["visium"].obs
st50_obs = st50_sdata.tables["visium"].obs

st48_grid_xy = st48_obs[["array_col", "array_row"]].values.astype(float)
st49_grid_xy = st49_obs[["array_col", "array_row"]].values.astype(float)
st50_grid_xy = st50_obs[["array_col", "array_row"]].values.astype(float)

st48_vals_raw = _dense_1d(st48_sdata.tables["visium"][:, gene].X).astype(float)
st49_vals_raw = _dense_1d(st49_sdata.tables["visium"][:, gene].X).astype(float)
st50_vals_raw = _dense_1d(st50_sdata.tables["visium"][:, gene].X).astype(float)


def _to_channel(values, channel):
    rgb = np.zeros((len(values), 3))
    vmax = np.nanpercentile(values, 99)
    rgb[:, color_idx[channel]] = np.clip(values / vmax, 0, 1) if vmax > 0 else values
    return rgb


fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(st48_grid_xy[:, 0], -st48_grid_xy[:, 1], c=_to_channel(st48_vals_raw, "red"), s=60, alpha=0.9)
ax.scatter(st49_grid_xy[:, 0], -st49_grid_xy[:, 1], c=_to_channel(st49_vals_raw, "green"), s=60, alpha=0.9)
ax.scatter(st50_grid_xy[:, 0], -st50_grid_xy[:, 1], c=_to_channel(st50_vals_raw, "blue"), s=60, alpha=0.9)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(f"{gene}: ST48 vs ST49 vs ST50 (array grid position -- no registration)")

legend_elements = [
    Patch(facecolor="red", label="ST48"),
    Patch(facecolor="green", label="ST49"),
    Patch(facecolor="blue", label="ST50"),
]
ax.legend(handles=legend_elements, loc="upper right", frameon=True)
plt.tight_layout()
plt.show()

## Bonus: 3D "stacked sheets" view -- the three sections aligned

Plot the three sections as flat sheets in 3D (one per `z` level), colored by the same gene.
Since ST48's and ST50's matched expression is already reindexed onto `merged`'s own spots
(Steps 3-4), all three layers share the exact same `(array_col, array_row)` grid position --
`merged`'s own Visium array grid coordinates -- there's nothing left to warp; a spot's x, y is
identical across all three sheets by construction, only its z (which section) and color (that
section's expression) change.

Rotate to look straight down the z-axis: matching high/low-expression regions should stack up
directly on top of each other across all three sheets.

In [ ]:
# On some machines, an old system-wide matplotlib install's mpl_toolkits shim registers a
# broken, version-mismatched mpl_toolkits in sys.modules before this notebook even starts
# (site.py runs it at interpreter startup), which also makes matplotlib.projections give up on
# registering the "3d" projection at its own (one-time) import. Evicting the stale cache entry
# and re-registering Axes3D by hand fixes both -- harmless if your environment doesn't have
# this issue.
import sys

for _name in list(sys.modules):
    if _name == "mpl_toolkits" or _name.startswith("mpl_toolkits."):
        del sys.modules[_name]

from mpl_toolkits.mplot3d import Axes3D

import matplotlib.projections as mprojections

if "3d" not in mprojections.get_projection_names():
    mprojections.register_projection(Axes3D)

In [ ]:
gene = "Mbp"
z_st48, z_st49, z_st50 = 0, 500, 1000

grid_xy = merged.obs[["array_col", "array_row"]].values.astype(float)
st48_vals_matched = merged.obsm["ST48_expression"][gene].values
st49_vals_matched = _dense_1d(merged[:, gene].X)
st50_vals_matched = merged.obsm["ST50_expression"][gene].fillna(0).values

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(grid_xy[:, 0], -grid_xy[:, 1], zs=z_st48, c=st48_vals_matched, cmap="viridis", s=4)
ax.scatter(grid_xy[:, 0], -grid_xy[:, 1], zs=z_st49, c=st49_vals_matched, cmap="viridis", s=4)
ax.scatter(grid_xy[:, 0], -grid_xy[:, 1], zs=z_st50, c=st50_vals_matched, cmap="viridis", s=4)
ax.set_zticks([z_st48, z_st49, z_st50])
ax.set_zticklabels(["ST48", "ST49", "ST50"])
ax.set_title(f"{gene} -- ST48 / ST49 / ST50 aligned, at merged's array grid position")

plt.tight_layout()
plt.show()